# rPPG Model Training Pipeline

Complete training workflow for the multi-metric rPPG model with confidence scoring.

In [ ]:
# Cell 1: Imports and configuration
import sys
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models.architecture import rPPGModel
from src.training.trainer import Trainer
from src.config import (
    DEVICE, FEATURE_DIM, HIDDEN_DIM, BATCH_SIZE,
    LEARNING_RATE, EPOCHS, CHECKPOINTS_DIR, RESULTS_DIR
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Create dummy data loaders for demonstration
# In production, replace with actual MCD-rPPG dataset

# Create dummy batch of video frames (B, T, C, H, W)
dummy_frames = torch.randn(64, 30, 3, 128, 128)  # 64 samples, 30 frames each
dummy_hr = torch.randint(60, 100, (64,)).float()  # Heart rate 60-100 bpm
dummy_rr = torch.randint(12, 20, (64,)).float()   # Respiratory rate 12-20
dummy_spo2 = torch.randint(95, 100, (64,)).float()  # SpO2 95-100%

# Create train and validation datasets
train_dataset = TensorDataset(dummy_frames[:50], dummy_hr[:50], dummy_rr[:50], dummy_spo2[:50])
val_dataset = TensorDataset(dummy_frames[50:], dummy_hr[50:], dummy_rr[50:], dummy_spo2[50:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Cell 3: Initialize model and trainer
device = torch.device(DEVICE)
model = rPPGModel(feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

trainer = Trainer(model, device, CHECKPOINTS_DIR, RESULTS_DIR)
print(f"Trainer initialized")
print(f"Checkpoint dir: {CHECKPOINTS_DIR}")
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
# Cell 4: Train model (using small number of epochs for demo)
# In production, set epochs=150
trainer.train(
    train_loader,
    val_loader,
    epochs=5,  # Demo: 5 epochs. Production: 150
    lr=LEARNING_RATE
)

In [ ]:
# Cell 5: Load and test best model
best_model = rPPGModel(feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM).to(device)
state_dict = torch.load(CHECKPOINTS_DIR / "final_model.pt")
best_model.load_state_dict(state_dict)

print("✓ Model trained and saved!")
print(f"Model path: {CHECKPOINTS_DIR / 'final_model.pt'}")

# Test inference on dummy data
best_model.eval()
with torch.no_grad():
    test_frames = dummy_frames[:1].to(device)
    hr_val, hr_conf, rr_val, rr_conf, spo2_val, spo2_conf = best_model(test_frames)

print("\nSample inference output:")
print(f"  HR: {hr_val.item():.1f} bpm (conf: {hr_conf.item():.3f})")
print(f"  RR: {rr_val.item():.1f} breaths/min (conf: {rr_conf.item():.3f})")
print(f"  SpO2: {spo2_val.item():.1f} % (conf: {spo2_conf.item():.3f})")

In [ ]:
# Cell 6: Evaluate on test set (placeholder)
import json

# In production, replace with actual test set evaluation
test_results = {
    "metrics": {
        "heart_rate": {"mae": 2.5, "rmse": 3.2, "correlation": 0.95},
        "respiratory_rate": {"mae": 1.5, "rmse": 2.0, "correlation": 0.92},
        "spo2": {"mae": 0.8, "rmse": 1.1, "correlation": 0.90}
    },
    "note": "These are placeholder values for demonstration. Use actual test set for real evaluation."
}

with open(RESULTS_DIR / "test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("Test Results Summary:")
for metric, scores in test_results["metrics"].items():
    print(f"\n{metric}:")
    for key, val in scores.items():
        print(f"  {key}: {val:.3f}")

In [ ]:
# Cell 7: Visualize training curves
import json
import matplotlib.pyplot as plt

# Load training logs
with open(RESULTS_DIR / "training_log.json", "r") as f:
    logs = json.load(f)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(logs["epoch"], logs["train_loss"], label="Training Loss", marker="o")
ax.plot(logs["epoch"], logs["val_loss"], label="Validation Loss", marker="s")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Progress")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.show()

print(f"Training curves saved to {RESULTS_DIR / 'training_curves.png'}")